In [1]:
# !pip install -q chromadb
# !pip install -q sentence-transformers
# !pip install -q gradio

In [2]:
import os
import re

import gradio as gr
import chromadb

from datetime import datetime
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from google import genai
from google.genai import types



from sentence_transformers import SentenceTransformer

In [3]:
load_dotenv(dotenv_path="local.env")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Security Error: GEMINI_API_KEY environment variable is missing!")

client = genai.Client()

In [4]:
OUTPUT_DIR = "mining_act_html_versions"
model = "gemma-4-31b-it"

In [5]:
print("Loading local embedding model ('all-MiniLM-L6-v2')...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading local embedding model ('all-MiniLM-L6-v2')...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
DB_STORAGE_PATH = "./chroma_db_storage"
os.makedirs(DB_STORAGE_PATH, exist_ok=True)

# 2. Bind Persistent Local Vector Client
chroma_client = chromadb.PersistentClient(path=DB_STORAGE_PATH)
collection_name = "latest_mining_act_rag"

# Fetch or spin up the collection database
collection = chroma_client.get_or_create_collection(name=collection_name)

print(f"Local Persistent Client connected at: '{DB_STORAGE_PATH}'")
print(f"Active collection profile context: '{collection_name}' (Current Size: {collection.count()} blocks)")
print("-" * 70)

Local Persistent Client connected at: './chroma_db_storage'
Active collection profile context: 'latest_mining_act_rag' (Current Size: 0 blocks)
----------------------------------------------------------------------


In [7]:
def find_latest_version(directory):
    if not os.path.exists(directory):
        raise FileNotFoundError(f"Directory '{directory}' is missing.")
    all_versions = []
    for filename in os.listdir(directory):
        if filename.startswith("Mining Act 1978") and filename.endswith(".html"):
            file_path = os.path.join(directory, filename)
            match = re.search(r"Mining Act 1978_.+?_(.+?)\.html", filename)
            if match:
                date_str = match.group(1).replace("_", " ")
                parsed_date = datetime.strptime(date_str, "%d %b %Y")
                all_versions.append({"path": file_path, "date": parsed_date})
    all_versions.sort(key=lambda x: x["date"], reverse=True)
    return all_versions[0]["path"]

def extract_clean_text(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    content_area = soup.find("section", id="main") or soup.find("section", id="outer")
    target_soup = content_area if content_area else soup
    for element in target_soup(["script", "style", "meta", "link", "header", "footer", "nav", "form"]):
        element.decompose()
    text = target_soup.get_text(separator="\n")
    return "\n".join([line.strip() for line in text.splitlines() if line.strip()])

def split_into_chunks(text, chunk_size=10000):
    # Smaller chunks (10k chars) are better optimized for local embedding semantic resolution
    chunks = []
    current_chunk = []
    current_size = 0
    for line in text.splitlines():
        current_chunk.append(line)
        current_size += len(line)
        if current_size >= chunk_size:
            chunks.append("\n".join(current_chunk))
            current_chunk = []
            current_size = 0
    if current_chunk:
        chunks.append("\n".join(current_chunk))
    return chunks

In [8]:
latest_file = find_latest_version(OUTPUT_DIR)
pure_text = extract_clean_text(latest_file)
document_chunks = split_into_chunks(pure_text)

DB_STORAGE_PATH = "./chroma_local_embed_storage"
chroma_client = chromadb.PersistentClient(path=DB_STORAGE_PATH)
collection = chroma_client.get_or_create_collection(name="mining_act_local_embeddings")

In [9]:
if collection.count() > 0:
    print(f"-> Local disk vector cache found with {collection.count()} blocks. Skipping embedding generation.")
else:
    print(f"Embedding {len(document_chunks)} chunks completely offline using local CPU/GPU...")
    for idx, chunk in enumerate(document_chunks):
        # Generate the embedding locally vector coordinates
        vector = embedding_model.encode(chunk).tolist()
        
        collection.add(
            embeddings=[vector],
            documents=[chunk],
            ids=[f"block_{idx}"]
        )
    print("-> Local database initialization complete and committed to disk!")

-> Local disk vector cache found with 54 blocks. Skipping embedding generation.


In [10]:
def ask_hybrid_rag(user_query: str):
    # Vectorize the question locally using the same local sentence-transformer model
    query_vector = embedding_model.encode(user_query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=20)
    retrieved_context = "\n\n---\n\n".join(results['documents'][0])
    
    rag_prompt = f"""
                    You are an expert legal assistant. 
                    Answer the user question using ONLY the provided verified context below.
                    If the context does not contain the answer, 
                    respond exactly with: "I don't know based on the provided context."
                    
                    ---
                    [VERIFIED CONTEXT]
                    {retrieved_context}
                    ---
                    
                    [QUESTION]
                    {user_query}
                """

    # print(rag_prompt)
    response = client.models\
                    .generate_content(
                                    model=model,
                                    contents=rag_prompt,
                                    config=types.GenerateContentConfig(
                                        temperature=0.0  
                                    )
                                )
    
    return response.text

In [11]:
# # Example Query 1
# question_1 = "What are the rules, in respect of the land or a part of the land the subject of the licence?"
# print(f"Question: {question_1}\n")
# print(ask_hybrid_rag(question_1))
# print("="*80)

In [12]:
def ask_hybrid_rag_with_memory(user_message: str, chat_history: list):
    """
    Processes chat conversations by vectorizing the latest question,
    assembling past chat memory strings, and prompting Gemini.
    """
    if not user_message.strip():
        return "Please enter a valid legal question."
        
    # 1. Vectorize the live question using your local sentence-transformer model
    query_vector = embedding_model.encode(user_message).tolist()
    
    # 2. Extract relevant legal documents context from ChromaDB [cite: 1436]
    results = collection.query(query_embeddings=[query_vector], n_results=10)
    retrieved_context = "\n\n---\n\n".join(results['documents'][0])
    
    # 3. Format the chat history buffer into a readable string for the model
    # Gradio chat_history format: [{"role": "user", "content": "text"}, {"role": "assistant", "content": "text"}]
    formatted_history = ""
    for turn in chat_history:
        role = "User" if turn["role"] == "user" else "Assistant"
        formatted_history += f"{role}: {turn['content']}\n"
        
    # 4. Construct the memory-grounded prompt template
    rag_prompt = f"""
                    You are an expert legal assistant specializing in the Western Australian Mining Act 1978. 
                    Your strict core rule is to answer the user question using ONLY the provided verified legal context below. 
                    
                    If the context does not contain the answer, or if a follow-up question refers to information 
                    outside of the provided context, respond exactly with: "I don't know based on the provided context." 
                    
                    ---
                    [VERIFIED LEGAL CONTEXT]
                    {retrieved_context}
                    ---
                    
                    [CONVERSATIONAL HISTORY]
                    {formatted_history}
                    ---
                    
                    [NEW USER QUESTION]
                    {user_message}
                    """

    # 5. Send the compiled context payload to Gemini
    response = client.models.generate_content(
        model=model,
        contents=rag_prompt, 
        config=types.GenerateContentConfig(
            temperature=0.0  # Keep at 0.0 to strictly prevent hallucinations
        )
    )
    
    return response.text

In [13]:
demo = gr.ChatInterface(
    fn=ask_hybrid_rag_with_memory, # Ensures compatibility with modern Gradio internal memory schemas
    title="⚖️ Mining Act 1978 Legal AI",
    description="Ask complex legislative questions. This assistant maintains conversational context and checks against local vector indices.",
    examples=[
        "What are the rules regarding the land subject to an exploration licence?",
        "Can you elaborate on the surrender requirements mentioned in your last answer?",
        "What is Part 4AA about?"
    ]
)

demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
